In [1]:
import math

import numpy as np
import numba
import numba.cuda as cuda

from RyR import *

In [2]:
RyR = np.random.rand(200,200,4).astype(np.float32)
RyR /= RyR.sum(axis=-1)[..., np.newaxis]
cp = 10 + 10 * np.random.rand(200,200).astype(np.float32) 

In [3]:
RyR[0,0,:]

array([0.30255017, 0.09311075, 0.29159972, 0.31273934], dtype=float32)

In [10]:
eps = 0.1
dt = 5e-3
nstep = 1
call_RyR_kernel(RyR, cp, eps, dt, nstep=100_000, threadsperblock=16)

In [11]:
k12 = cp[0,0]**2

log_cp_K = math.log(cp[0,0]) - math.log(10.0)
hill_fn = 1.0 / (1.0 + math.exp(23 * log_cp_K))
Mhat = (math.sqrt(1.0 + 8 * hill_fn * 1.0) - 1.0) / (4 * hill_fn * 1.0)

k14 = Mhat

k21 = 1.0
k23 = Mhat
k43 = cp[0,0]**2
k41 = 1.0
k34 = 1.0
k32 = k41*k12/k43

In [12]:
pi1 = 1.0
pi2 = k12 / k21
pi3 = k23 / k32 * pi2
pi4 = k34 / k43 * pi3

In [13]:
sum_pi = pi1 + pi2 + pi3 + pi4
pi1 /= sum_pi
pi2 /= sum_pi
pi3 /= sum_pi
pi4 /= sum_pi

In [14]:
[pi1, pi2, pi3, pi4]

[0.0029505945427406514,
 0.4982465614055169,
 0.495866345047118,
 0.0029364990046242974]

In [15]:
RyR[0,0,:]

array([0.0111071 , 0.5084529 , 0.47215086, 0.00828916], dtype=float32)